In [ ]:
"""
============================================================
Freedom Insurance - ОГПО Scoring Model Pipeline v6 FINAL
============================================================

============================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import brentq

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import joblib

# ──────────────────────────────────────────────
# 0. ПУТИ
# ──────────────────────────────────────────────
DATA_DIR   = Path("/Users//Desktop/final_dataset")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE  = DATA_DIR / "test_final.csv"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_LR = 0.70

# ──────────────────────────────────────────────
# 1. ЗАГРУЗКА
# ──────────────────────────────────────────────
print("=" * 60)
print("1. Загрузка данных")
train = pd.read_csv(TRAIN_FILE, low_memory=False)
test  = pd.read_csv(TEST_FILE,  low_memory=False)
print(f"Train: {train.shape} | Test: {test.shape}")

# ──────────────────────────────────────────────
# 1b. HOLDOUT SPLIT
# ──────────────────────────────────────────────
# оставить часть train для проверки LR.

from sklearn.model_selection import train_test_split as tts
train, train_holdout = tts(train, test_size=0.20, random_state=42)
train         = train.reset_index(drop=True)
train_holdout = train_holdout.reset_index(drop=True)
print(f"  Train fit: {len(train):,} | Holdout (20%): {len(train_holdout):,}")

# ──────────────────────────────────────────────
# 2. ДЕДУПЛИКАЦИЯ
# ──────────────────────────────────────────────
score_cols   = [c for c in train.columns if c.startswith("SCORE")]
drop_for_dup = ["unique_id"] + score_cols
dup_mask = train.drop(columns=drop_for_dup, errors="ignore").duplicated()
print(f"\n2. Дубликатов: {dup_mask.sum()} -> убраны")
train = train[~dup_mask].reset_index(drop=True)

# ── Автоопределение года датасета из operation_date ───────────────────────

DATASET_YEAR = int(
    pd.to_datetime(train["operation_date"], errors="coerce")
    .dt.year.dropna().mode()[0]
)
print(f"  Определён год датасета: {DATASET_YEAR}")

# Удаляем SCORE-колонки где >95% нулей
score_cols_num  = train[score_cols].apply(pd.to_numeric, errors="coerce")
score_zero_rate = (score_cols_num == 0).mean()
score_cols_drop = score_zero_rate[score_zero_rate > 0.95].index.tolist()
score_cols      = [c for c in score_cols if c not in score_cols_drop]
print(f"  SCORE-колонок удалено (>95% нулей): {len(score_cols_drop)}")
print(f"  SCORE-колонок осталось: {len(score_cols)}")

# SCORE агрегация: 128 -> 12 групповых средних
score_groups = {}
for col in score_cols:
    parts = col.split("_")
    prefix = "_".join(parts[:2])
    score_groups.setdefault(prefix, []).append(col)

for df in [train, test]:
    for prefix, cols in score_groups.items():
        df[f"{prefix}_avg"] = df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
score_avg_cols = [f"{p}_avg" for p in score_groups]

train.drop(columns=score_cols, inplace=True)
test.drop(columns=[c for c in score_cols if c in test.columns], inplace=True)
score_cols = score_avg_cols
print(f"  SCORE сжаты: {sum(len(v) for v in score_groups.values())} -> {len(score_cols)} групповых средних")

# ──────────────────────────────────────────────
# 3. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ
# ──────────────────────────────────────────────
train["claim_amount"] = pd.to_numeric(train["claim_amount"], errors="coerce").fillna(0)
train["claim_cnt"]    = pd.to_numeric(train["claim_cnt"],    errors="coerce").fillna(0)

if "is_claim" not in train.columns or train["is_claim"].isna().all():
    train["is_claim"] = (train["claim_amount"] > 0).astype(int)
else:
    train["is_claim"] = train["is_claim"].fillna(0).astype(int)

for col in ["claim_amount", "claim_cnt"]:
    if col in test.columns:
        test[col] = pd.to_numeric(test[col], errors="coerce").fillna(0)

CLAIM_RATE = train["is_claim"].mean()
print(f"\n3. Частота выплат: {CLAIM_RATE:.3%}")

# ──────────────────────────────────────────────
# 4. ОЧИСТКА ПРИЗНАКОВ
# ──────────────────────────────────────────────
PROTECTED_DROP = {
    "claim_amount", "claim_cnt", "is_claim",
    "premium", "premium_wo_term",
    "contract_number", "unique_id",
    "driver_iin", "operation_date", "region_id",
    "vehicle_type_id", "bonus_malus",
    "experience_year", "car_year", "engine_power", "engine_volume",
}
miss_pct  = train.isnull().mean() * 100
drop_cols = [c for c in miss_pct[miss_pct > 80].index
             if c not in PROTECTED_DROP and not c.startswith("SCORE")]
if drop_cols:
    train.drop(columns=drop_cols, inplace=True)
    test.drop(columns=[c for c in drop_cols if c in test.columns], inplace=True)
print(f"\n4a. Удалено колонок >80% пропусков: {len(drop_cols)}  {drop_cols}")

def clean_bonus_malus(val):
    # Шкала по закону РК (от худшего к лучшему):
    # M2(-3) -> M1(-2) -> M(-1) -> 0 -> 1 -> 2 -> ... -> 13
    # Класс 3 - дефолт для новых водителей без истории
    s = str(val).strip()
    if s in ("", "nan", "None"):   
        return 3
    su = s.upper()
    if su == "M2": return -3       # самый штрафной класс
    if su == "M1": return -2
    if su == "M":  return -1       # штрафной класс
    try:
        v = int(s)
        return v if 0 <= v <= 13 else 3  
    except ValueError:
        return 3

def clean_experience_year(val):
    try:
        v = float(val)
    except (ValueError, TypeError):
        return np.nan
    if v < 0:
        return np.nan
    if 1900 <= v <= DATASET_YEAR:
        return max(0, DATASET_YEAR - int(v))
    if v > 70:
        return np.nan
    return v

def clean_car_year(val):
    try:
        s = str(val).strip().replace(" ", "")
        v = int(float(s))
    except (ValueError, TypeError):
        return np.nan
    if 1950 <= v <= DATASET_YEAR:   
        return v
    if v < 100:
        c2 = 2000 + v
        c1 = 1900 + v
        if 1990 <= c2 <= DATASET_YEAR:
            return c2
        if 1950 <= c1 <= DATASET_YEAR:
            return c1
    return np.nan

def parse_car_age(val):
    v = str(val).strip().lower()
    if "до 7" in v:    return 0
    if "свыше 7" in v: return 1
    return pd.to_numeric(val, errors="coerce")

for df in [train, test]:
    df["bonus_malus_clean"] = df["bonus_malus"].apply(clean_bonus_malus)
    df["experience_year"]   = df["experience_year"].apply(clean_experience_year)
    df["car_year"]          = df["car_year"].apply(clean_car_year)
    df["car_age"]           = df["car_age"].apply(parse_car_age)

    df["engine_volume"] = pd.to_numeric(df["engine_volume"], errors="coerce")
    df["engine_power"]  = pd.to_numeric(df["engine_power"],  errors="coerce")
    df.loc[df["engine_volume"] < 0.3,  "engine_volume"] = np.nan
    df.loc[df["engine_volume"] > 10,   "engine_volume"] = np.nan
    df.loc[df["engine_power"]  < 5,    "engine_power"]  = np.nan
    df.loc[df["engine_power"]  > 1000, "engine_power"]  = np.nan
    df["engine_volume"] = df["engine_volume"].fillna(df["engine_volume"].median())
    df["engine_power"]  = df["engine_power"].fillna(df["engine_power"].median())

    df["premium"] = pd.to_numeric(df["premium"], errors="coerce")
    df.loc[df["premium"] <= 0, "premium"] = np.nan
    df["premium"] = df["premium"].fillna(df["premium"].median())

    
    df["premium_wo_term"] = pd.to_numeric(df["premium_wo_term"], errors="coerce")
    df["premium_wo_term"] = df["premium_wo_term"].fillna(df["premium"] * 0.85)

train["claim_amount"] = pd.to_numeric(train["claim_amount"], errors="coerce").fillna(0)
bad_claim = (train["is_claim"] == 1) & (train["claim_amount"] == 0)
if bad_claim.sum() > 0:
    med_claim = train.loc[(train["is_claim"] == 1) & (train["claim_amount"] > 0),
                          "claim_amount"].median()
    train.loc[bad_claim, "claim_amount"] = med_claim
    print(f"  claim_amount: заполнено {bad_claim.sum()} строк медианой {med_claim:,.0f} тг")

print("4. Очистка признаков - готово")
print(f"   Пропусков после чистки: {train.isnull().sum().sum():,}")

# ──────────────────────────────────────────────
# 5. ПОЛИСНЫЙ УРОВЕНЬ - LR ДО
# ──────────────────────────────────────────────
def policy_agg(df, extra=None):
    agg = dict(
        premium        =("premium",        "first"),
        premium_wo_term=("premium_wo_term","first"),
        claim_amount   =("claim_amount",   "first"),
        is_claim       =("is_claim",       "max"),
    )
    if extra:
        agg.update(extra)
    return df.groupby("contract_number", as_index=False).agg(**agg)

train_pol = policy_agg(train)
train_pol["prem_base"] = np.where(
    train_pol["premium_wo_term"] > 0,
    train_pol["premium_wo_term"],
    train_pol["premium"]
)

LR_BEFORE = train_pol["claim_amount"].sum() / train_pol["prem_base"].sum()
print(f"\n5. LR ДО: {LR_BEFORE:.2%} | Полисов: {len(train_pol):,}")

# ──────────────────────────────────────────────
# 6. FEATURE ENGINEERING
# ──────────────────────────────────────────────
def feature_engineering(df):
    df = df.copy()
    df["operation_date"] = pd.to_datetime(df["operation_date"], errors="coerce")
    df["policy_year"]    = df["operation_date"].dt.year.fillna(DATASET_YEAR).astype(int)
    df["policy_month"]   = df["operation_date"].dt.month.fillna(1).astype(int)
    df["policy_quarter"] = ((df["policy_month"] - 1) // 3 + 1).astype(int)

    df["car_age_calc"] = (df["policy_year"] - df["car_year"]).clip(0, 50)

    # КБМ признаки
    # Диапазон bonus_malus_clean: от -3 (M2, худший) до 13 (лучший)
    # bm_risk = 1.0 для M2, 0.0 для класса 13
    # Формула: (13 - class) / (13 - (-3)) = (13 - class) / 16
    df["bm_risk"]      = (13 - df["bonus_malus_clean"]) / 16.0
    df["bm_is_base"]   = (df["bonus_malus_clean"] == 3).astype(int)   # новый водитель
    df["bm_is_good"]   = (df["bonus_malus_clean"] >= 8).astype(int)   # хороший водитель
    df["bm_is_bad"]    = (df["bonus_malus_clean"] <= 0).astype(int)   # штрафной (0,M,M1,M2)
    df["bm_high_risk"] = (df["bonus_malus_clean"] <= 3).astype(int)   # повышенный риск
    df["bm_low_risk"]  = (df["bonus_malus_clean"] >= 10).astype(int)  # низкий риск

    # Стаж
    df["exp_group"] = pd.cut(
        df["experience_year"],
        bins=[-1, 0, 2, 5, 10, 20, 100],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype("float")
    df["is_new_driver"] = (df["experience_year"] <= 2).astype(float)
    df["exp_novice"]    = (df["experience_year"] <= 1).astype(int)

    # Двигатель
    df["power_per_vol"]   = df["engine_power"] / (df["engine_volume"] + 1)
    df["power_per_liter"] = (df["engine_power"] /
                             df["engine_volume"].replace(0, np.nan)).fillna(0)
    df["high_power"] = (df["engine_power"] > 150).astype(float)

    # Страхователь
    df["is_individual_person"] = pd.to_numeric(df["is_individual_person"],
                                               errors="coerce").fillna(1)
    df["is_residence"] = pd.to_numeric(df["is_residence"],
                                       errors="coerce").fillna(1)

    # Взаимодействия
    df["bm_x_exp"]    = df["bm_risk"] * df["experience_year"].fillna(0)
    df["power_x_new"] = df["engine_power"].fillna(0) * df["is_new_driver"]
    df["risk_score"]  = (df["bm_risk"] / (df["experience_year"].fillna(0) + 1)).round(4)

    # Премия - только log_premium (известна при оформлении полиса)


    # Взаимодействия с регионом и типом ТС
    df["region_x_bm"]     = pd.to_numeric(df["region_id"],
                                           errors="coerce").fillna(0) * df["bm_risk"]
    df["vehicle_x_power"] = pd.to_numeric(df["vehicle_type_id"],
                                           errors="coerce").fillna(0) * df["power_per_vol"]
    return df

train = feature_engineering(train)
test  = feature_engineering(test)
print("6. Feature Engineering - готово")



# ──────────────────────────────────────────────
# 6c. АГРЕГАТЫ ПО ПОЛИСУ (OOF-safe)
# ──────────────────────────────────────────────
print("6c. Агрегаты по contract_number")

policy_feat_agg = (
    train.groupby("contract_number", as_index=False)
         .agg(
             pol_driver_cnt  = ("driver_iin",         "nunique"),
             pol_avg_bm      = ("bonus_malus_clean",  "mean"),
             pol_max_bm_risk = ("bm_risk",            "max"),
             pol_min_exp     = ("experience_year",    "min"),
             pol_any_new     = ("is_new_driver",      "max"),
         )
)
train = train.merge(policy_feat_agg, on="contract_number", how="left")
test  = test.merge(policy_feat_agg,  on="contract_number", how="left")
for col in ["pol_driver_cnt","pol_avg_bm","pol_max_bm_risk","pol_min_exp","pol_any_new"]:
    fill_val = policy_feat_agg[col].mean()
    train[col] = train[col].fillna(fill_val)
    test[col]  = test[col].fillna(fill_val)
print("  pol_driver_cnt, pol_avg_bm, pol_max_bm_risk, pol_min_exp, pol_any_new")

# ──────────────────────────────────────────────
# 7. WoE / IV
# ──────────────────────────────────────────────
def calc_iv(df, feature, target, bins=10):
    tmp = df[[feature, target]].copy().dropna()
    if len(tmp) < 100:
        return 0.0
    if tmp[feature].dtype in ["object", "category"]:
        tmp["bucket"] = tmp[feature].astype(str)
    else:
        try:
            tmp["bucket"] = pd.qcut(tmp[feature], q=bins, duplicates="drop")
        except Exception:
            tmp["bucket"] = tmp[feature].astype(str)
    te  = tmp[target].sum()
    tne = len(tmp) - te
    g   = tmp.groupby("bucket")[target].agg(["sum", "count"])
    g.columns = ["events", "count"]
    g["ne"]  = g["count"] - g["events"]
    g["pe"]  = g["events"] / (te  + 1e-9)
    g["pne"] = g["ne"]    / (tne + 1e-9)
    g["woe"] = np.log(g["pe"] / (g["pne"] + 1e-9) + 1e-9)
    g["iv"]  = (g["pe"] - g["pne"]) * g["woe"]
    return g["iv"].sum()


iv_feats = [
    "bonus_malus_clean","bm_risk","bm_is_good","bm_is_bad",
    "bm_high_risk","bm_low_risk",
    "experience_year","exp_group","is_new_driver","exp_novice",
    "car_age","car_age_calc","car_year",
    "engine_power","engine_volume","power_per_vol","power_per_liter","high_power",
    "is_individual_person","is_residence",
    "region_id","vehicle_type_id","policy_month","policy_quarter",
    "bm_x_exp","power_x_new","risk_score",
    "pol_driver_cnt","pol_avg_bm","pol_max_bm_risk","pol_min_exp","pol_any_new",
    "region_x_bm","vehicle_x_power",
]
iv_df = pd.DataFrame([
    {"feature": f, "IV": round(calc_iv(train, f, "is_claim"), 4)}
    for f in iv_feats if f in train.columns
]).sort_values("IV", ascending=False)
print("\n7. IV:\n", iv_df.to_string(index=False))
iv_df.to_csv(OUTPUT_DIR / "iv_results.csv", index=False)

# ──────────────────────────────────────────────
# 8. ПРИЗНАКИ ДЛЯ МОДЕЛИ
# ──────────────────────────────────────────────
base_features = [
    # КБМ - оставляем оба: bm_risk (непрерывный) и bonus_malus_clean (целочисленный)
    # дают разные сплиты в деревьях несмотря на высокую корреляцию
    "bonus_malus_clean","bm_risk",
    # Стаж - убраны is_new_driver (IV=0), exp_novice (IV=0)
    "experience_year","exp_group",
    # Авто - car_year убран (корреляция 0.9999 с car_age_calc)
    "car_age","car_age_calc",
    # Двигатель - engine_power убран (корреляция 0.9999 с power_per_vol)
    # power_per_liter убран ранее (корреляция 0.9999 с power_per_vol)
    # engine_volume и high_power убраны (IV=0)
    "power_per_vol",
    # Страхователь - убраны is_individual_person, is_residence (IV=0)
    # Гео / дата - убран policy_quarter (IV=0.0014, почти ноль)
    "region_id","vehicle_type_id",
    "policy_year","policy_month",
    # Премия - только log_premium (известна при оформлении)
    # Агрегаты по полису - OOF-safe, утечки таргета нет
    "pol_driver_cnt","pol_avg_bm","pol_max_bm_risk","pol_min_exp",
    # Взаимодействия
    "region_x_bm","vehicle_x_power",
]
feature_cols = [f for f in base_features if f in train.columns] + score_cols

cat_cols = ["region_id", "vehicle_type_id"]
le_dict  = {}
for col in cat_cols:
    if col in train.columns and col in test.columns:
        le = LabelEncoder()
        combined = train[col].astype(str).tolist() + test[col].astype(str).tolist()
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))
        le_dict[col] = le

POL_AGG_COLS = ["contract_number", "driver_iin",
                "bonus_malus_clean", "bm_risk", "experience_year", "is_new_driver"]
train_meta = train[POL_AGG_COLS].copy()

def compute_pol_features(meta_df):
    """Вычисляет pol_* агрегаты по переданному подмножеству строк."""
    return (
        meta_df.groupby("contract_number", as_index=False)
               .agg(
                   pol_driver_cnt  = ("driver_iin",         "nunique"),
                   pol_avg_bm      = ("bonus_malus_clean",  "mean"),
                   pol_max_bm_risk = ("bm_risk",            "max"),
                   pol_min_exp     = ("experience_year",    "min"),
                   pol_any_new     = ("is_new_driver",      "max"),
               )
    )

POL_FEAT_NAMES = ["pol_driver_cnt","pol_avg_bm","pol_max_bm_risk","pol_min_exp"]

X      = train[feature_cols].apply(pd.to_numeric, errors="coerce")
y_freq = train["is_claim"]
X_test = test[feature_cols].apply(pd.to_numeric, errors="coerce")

# ── Корреляционный анализ перед обучением ─────────────────────────────────
print("\n8b. Корреляционный анализ признаков (порог > 0.85):")
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
    .query("correlation > 0.85")
    .sort_values("correlation", ascending=False)
)
if len(high_corr) == 0:
    print("  ✅ Высококоррелированных пар не найдено")
else:
    print(f"  Найдено пар: {len(high_corr)}")
    print(high_corr.to_string(index=False))

pol_feat_pos = {c: feature_cols.index(c) for c in POL_FEAT_NAMES if c in feature_cols}

mask_claim = train["is_claim"] == 1
X_sev = X[mask_claim]
y_sev = np.log1p(train.loc[mask_claim, "claim_amount"])

print(f"\n8. Features: {len(feature_cols)} | Claims: {mask_claim.sum():,}")

# ──────────────────────────────────────────────
# 9. FREQUENCY MODEL - 5-FOLD CV
# ──────────────────────────────────────────────
print("\n9. Frequency Model (5-fold CV)")

SPW = (y_freq == 0).sum() / (y_freq == 1).sum()

params_freq = {
    "objective":         "binary",
    "metric":            "auc",
    "learning_rate":     0.01,
    "num_leaves":        20,
    "max_depth":         4,
    "min_child_samples": 1200,
    "feature_fraction":  0.6,
    "bagging_fraction":  0.7,
    "bagging_freq":      1,
    "lambda_l1":         5.0,
    "lambda_l2":         12.0,
    "min_split_gain":    1.0,
    "extra_trees":       True,   # доп. рандомизация -> сглаживает разрыв holdout LR
    "path_smooth":       1.0,    # сглаживание предсказаний на редких значениях признаков
    "scale_pos_weight":  SPW,
    "verbose":           -1,
    "n_jobs":            1,
    "random_state":      42,
}

# params_freq = {
#     "objective":         "binary",
#     "metric":            "auc",
#     "learning_rate":     0.01,
#     "num_leaves":        15,
#     "max_depth":         4,
#     "min_child_samples": 1500,
#     "feature_fraction":  0.6,
#     "bagging_fraction":  0.7,
#     "bagging_freq":      1,
#     "lambda_l1":         5.0,
#     "lambda_l2":         12.0,
#     "min_split_gain":    1.5,
#     "extra_trees":       True,   # доп. рандомизация -> сглаживает разрыв holdout LR
#     "path_smooth":       1.0,    # сглаживание предсказаний на редких значениях признаков
#     "scale_pos_weight":  SPW,
#     "verbose":           -1,
#     "n_jobs":            1,
#     "random_state":      42,
# }

# Monotonic constraints: bm_risk и bonus_malus_clean должны монотонно
# увеличивать вероятность ДТП - физически обосновано, снижает переобучение.
# +1 = монотонный рост, -1 = монотонное убывание, 0 = без ограничения
_mc = {f: 0 for f in feature_cols}
_mc["bm_risk"]          = +1   # выше риск КБМ -> выше вероятность
_mc["bonus_malus_clean"] = -1  # выше номер КБМ (лучше водитель) -> ниже вероятность
_mc["pol_avg_bm"]        = -1  # выше средний КБМ полиса -> ниже вероятность
_mc["pol_max_bm_risk"]   = +1  # выше макс. риск полиса -> выше вероятность
params_freq["monotone_constraints"] = [_mc[f] for f in feature_cols]
params_freq["monotone_constraints_method"] = "advanced"  # лучше соблюдает ограничения

# StratifiedGroupKFold - все строки одного contract_number
# всегда в одном фолде -> устраняет утечку через общий полис
from sklearn.model_selection import StratifiedGroupKFold
skf       = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
groups    = train["contract_number"].values  # группировка по полису
oof_prob      = np.zeros(len(X))
test_prob     = np.zeros(len(X_test))
auc_list      = []
auc_train_list= []
best_iters    = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_freq, groups=groups)):

    # OOF-агрегация pol_* только по train-части фолда
    fold_train_meta = train_meta.iloc[tr_idx]
    fold_agg        = compute_pol_features(fold_train_meta)
    global_means    = {c: fold_agg[c].mean() for c in POL_FEAT_NAMES}
    fold_agg_map    = fold_agg.set_index("contract_number")[POL_FEAT_NAMES]

    def get_pol_vals(idx_arr):
        contracts = train_meta.iloc[idx_arr]["contract_number"].values
        rows = fold_agg_map.reindex(contracts)
        for col in POL_FEAT_NAMES:
            rows[col] = rows[col].fillna(global_means[col])
        return rows[POL_FEAT_NAMES].values

    X_tr_arr  = X.iloc[tr_idx].values.copy()
    X_val_arr = X.iloc[val_idx].values.copy()
    pol_tr    = get_pol_vals(tr_idx)
    pol_val   = get_pol_vals(val_idx)

    for i, col in enumerate(POL_FEAT_NAMES):
        if col in pol_feat_pos:
            j = pol_feat_pos[col]
            X_tr_arr[:, j]  = pol_tr[:, i]
            X_val_arr[:, j] = pol_val[:, i]

    X_tr_fold  = pd.DataFrame(X_tr_arr,  columns=feature_cols)
    X_val_fold = pd.DataFrame(X_val_arr, columns=feature_cols)
    y_tr, y_val = y_freq.iloc[tr_idx], y_freq.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr_fold,  label=y_tr)
    dval   = lgb.Dataset(X_val_fold, label=y_val, reference=dtrain)

    m = lgb.train(
        params_freq, dtrain,
        num_boost_round=2000,
        valid_sets=[dtrain, dval],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(200, verbose=False),
            lgb.log_evaluation(200),
        ],
    )
    oof_prob[val_idx] = m.predict(X_val_fold)
    test_prob        += m.predict(X_test) / 5

    auc_val   = roc_auc_score(y_val, oof_prob[val_idx])
    auc_train = roc_auc_score(y_tr,  m.predict(X_tr_fold))

    auc_list.append(auc_val)
    auc_train_list.append(auc_train)
    best_iters.append(m.best_iteration)

    print(f"  Fold {fold+1}: AUC_train={auc_train:.4f} | AUC_val={auc_val:.4f} | "
          f"GAP={auc_train-auc_val:+.4f} | best_iter={m.best_iteration}")

print(f"\n  CV AUC (val):   {np.mean(auc_list):.4f} ± {np.std(auc_list):.4f}")
print(f"  CV AUC (train): {np.mean(auc_train_list):.4f} ± {np.std(auc_train_list):.4f}")
print(f"  GINI:           {2*np.mean(auc_list)-1:.4f}")
mean_gap = np.mean(auc_train_list) - np.mean(auc_list)
print(f"  Train-val GAP:  {mean_gap:+.4f}")

if mean_gap < 0.05:
    print("  ✅ Переобучения нет (gap < 0.05)")
elif mean_gap < 0.10:
    print("  ✅ Допустимый GAP (0.05–0.10)")
elif mean_gap < 0.15:
    print("  ⚠️  GAP 0.10–0.15 - повышенный, приемлемый")
else:
    print("  ❌ GAP > 0.15 - высокий")

# Финальная модель
model_freq = lgb.train(
    params_freq,
    lgb.Dataset(X, label=y_freq),
    num_boost_round=int(np.mean(best_iters)),
    callbacks=[lgb.log_evaluation(500)],
)

# Калибровка вероятностей
def calibrate(probs, true_rate):
    logits = np.log(np.clip(probs, 1e-9, 1-1e-9) / (1 - np.clip(probs, 1e-9, 1-1e-9)))
    def mean_diff(b):
        return (1 / (1 + np.exp(-(logits + b)))).mean() - true_rate
    bias = brentq(mean_diff, -20, 20)
    return 1 / (1 + np.exp(-(logits + bias)))

train["prob_claim"] = calibrate(oof_prob,  CLAIM_RATE)
test["prob_claim"]  = calibrate(test_prob, CLAIM_RATE)
print(f"\n  Калибровка: train={train['prob_claim'].mean():.4f}, "
      f"test={test['prob_claim'].mean():.4f} (цель={CLAIM_RATE:.4f})")

# ── F1-score  ────────────────────────────────────────────
# Для несбалансированных данных (1.944% положительных) порог = CLAIM_RATE
# даёт более информативный F1 чем стандартный порог 0.5
from sklearn.metrics import f1_score, classification_report

threshold_f1 = CLAIM_RATE  # 0.0194 - равен реальной частоте ДТП
y_pred_binary = (train["prob_claim"] >= threshold_f1).astype(int)

f1 = f1_score(y_freq, y_pred_binary)
print(f"\n  F1-score (порог={threshold_f1:.4f}): {f1:.4f}")
print(f"  Примечание: низкий F1 ожидаем при дисбалансе классов 1:51")
print(f"\n  Classification Report:")
print(classification_report(y_freq, y_pred_binary,
                            target_names=["No claim", "Claim"],
                            digits=4))

# ──────────────────────────────────────────────
# 10. SEVERITY MODEL
# ──────────────────────────────────────────────
print("\n10. Severity Model")

X_str, X_sval, y_str, y_sval = train_test_split(
    X_sev, y_sev, test_size=0.2, random_state=42
)

params_sev = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.01,          # было 0.03 → медленнее но точнее
    "num_leaves":        127,           # было 20 → главное изменение
    "max_depth":         -1,            # было 4 → без ограничения
    "min_child_samples": 10,            # было 50 → меньше ограничение
    "feature_fraction":  0.7,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "lambda_l1":         1.0,
    "lambda_l2":         0.5,           # добавить небольшую L2
    "verbose":           -1,
    "n_jobs":            1,
    "random_state":      42,
}
# num_boost_round оставить 6000, early_stopping 200 — всё найдёт сам

model_sev = lgb.train(
    params_sev,
    lgb.Dataset(X_str, label=y_str),
    num_boost_round=6000,
    valid_sets=[lgb.Dataset(X_sval, label=y_sval)],
    callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(500)],
)

r2 = r2_score(y_sval, model_sev.predict(X_sval))
print(f"  R²: {r2:.4f}")

train["pred_log_severity"] = model_sev.predict(X)
test["pred_log_severity"]  = model_sev.predict(X_test)

actual_log_mean = y_sev.mean()
pred_log_mean   = train.loc[mask_claim, "pred_log_severity"].mean()
log_bias        = actual_log_mean - pred_log_mean
print(f"  Log-bias коррекция: {log_bias:.4f}")

train["expected_severity"] = np.expm1(train["pred_log_severity"] + log_bias)
test["expected_severity"]  = np.expm1(test["pred_log_severity"]  + log_bias)

min_sev = np.expm1(y_sev.median()) * 0.1
train["expected_severity"] = train["expected_severity"].clip(lower=min_sev)
test["expected_severity"]  = test["expected_severity"].clip(lower=min_sev)

# ── Региональный поправочный коэффициент severity ─────────────────────────
# Считаем на train: отношение реальной средней тяжести к предсказанной
# по каждому региону. Ограничиваем в [0.5, 2.0] чтобы не было экстремальных сдвигов.
# Применяем только к регионам с >= 10 страховых случаев (иначе ненадёжно).
print("\n10b. Региональный поправочный коэффициент severity:")
claims_train = train[train["is_claim"] == 1].copy()
region_sev_stats = (
    claims_train.groupby("region_id")
    .agg(
        actual_sev   = ("claim_amount",       "mean"),
        pred_sev     = ("expected_severity",  "mean"),
        n_claims     = ("claim_amount",       "count"),
    )
    .reset_index()
)
region_sev_stats = region_sev_stats[region_sev_stats["n_claims"] >= 10].copy()
region_sev_stats["region_factor"] = (
    region_sev_stats["actual_sev"] / region_sev_stats["pred_sev"]
).clip(0.5, 2.0)

region_factor_map = region_sev_stats.set_index("region_id")["region_factor"].to_dict()

print(f"  Регионов с поправкой: {len(region_factor_map)}")
print(f"  {'region_id':<12} {'factor':>10} {'n_claims':>10}")
print("  " + "─" * 36)
for rid, row_r in region_sev_stats.sort_values("region_factor", ascending=False).iterrows():
    flag = "⚠️ завышен" if row_r["region_factor"] > 1.5 else (
           "⬇️ занижен" if row_r["region_factor"] < 0.7 else "✅")
    print(f"  {int(row_r['region_id']):<12} {row_r['region_factor']:>10.3f} "
          f"{int(row_r['n_claims']):>10}  {flag}")

# Применяем к train и test
def apply_region_factor(df, factor_map, global_f=1.0):
    factors = df["region_id"].map(factor_map).fillna(global_f)
    return df["expected_severity"] * factors

train["expected_severity"] = apply_region_factor(train, region_factor_map)
test["expected_severity"]  = apply_region_factor(test,  region_factor_map)

# Повторно клипируем после поправки
train["expected_severity"] = train["expected_severity"].clip(lower=min_sev)
test["expected_severity"]  = test["expected_severity"].clip(lower=min_sev)

print(f"  Средняя ожидаемая тяжесть (train): {train['expected_severity'].mean():,.0f} тг")

# ──────────────────────────────────────────────
# 11. РАСЧЁТ НОВОЙ ПРЕМИИ
# ──────────────────────────────────────────────
print("\n11. Расчёт новой премии (итеративная калибровка)")

train["expected_loss"] = train["prob_claim"] * train["expected_severity"]

train_pol = policy_agg(
    train,
    extra=dict(
        expected_loss = ("expected_loss", "first"),
        prob_claim    = ("prob_claim",    "first"),
    )
)

# prem_base: premium_wo_term используется ТОЛЬКО здесь - для оценки LR
# Это допустимо: оценка рентабельности
train_pol["prem_base"] = np.where(
    train_pol["premium_wo_term"] > 0,
    train_pol["premium_wo_term"],
    train_pol["premium"]
)

total_claims = train_pol["claim_amount"].sum()
print(f"  Итого выплат: {total_claims:,.0f} тг")
print(f"  Нужная суммарная премия при LR=70%: {total_claims/TARGET_LR:,.0f} тг")

def apply_pricing(df, scale):
    raw         = df["expected_loss"] * scale / TARGET_LR
    upper_bound = df["premium"] * 3    # ограничение: не более чем в 3 раза
    lower_bound = df["premium"] * 0.0  # ограничение: снижение не более 100% (= 0)
    capped      = raw.clip(lower=lower_bound, upper=upper_bound)
    ratio       = capped / (df["premium"] + 1e-9)
    pbase_new   = df["prem_base"] * ratio
    lr          = df["claim_amount"].sum() / (pbase_new.sum() + 1e-9)
    return lr, capped, pbase_new

scale = total_claims / (TARGET_LR * train_pol["expected_loss"].sum() + 1e-9)

for iteration in range(30):
    lr_cur, _, _ = apply_pricing(train_pol, scale)
    print(f"  iter {iteration+1:2d}: scale={scale:.4f}  ->  LR={lr_cur:.2%}")
    if abs(lr_cur - TARGET_LR) < 0.0005:
        break
    scale *= (lr_cur / TARGET_LR)

lr_final, train_pol["premium_new"], train_pol["prem_base_new"] = apply_pricing(
    train_pol, scale
)

GROUP1_THRESHOLD = 1.40  
train_pol["group"] = np.where(
    train_pol["premium_new"] <= train_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)

# ──────────────────────────────────────────────
# 11b. КАЛИБРОВКА ПО ГРУППАМ
# ──────────────────────────────────────────────
print("\n11b. Калибровка по группам (цель: LR~70% в каждой)")

for recalib_round in range(2):
    for grp in [1, 2]:
        mask_g = train_pol["group"] == grp
        sub    = train_pol[mask_g]
        if len(sub) == 0:
            continue
        lr_g    = sub["claim_amount"].sum() / sub["prem_base_new"].sum()
        adj     = lr_g / TARGET_LR
        new_raw = sub["premium_new"] * adj
        upper   = sub["premium"] * 3
        train_pol.loc[mask_g, "premium_new"] = new_raw.clip(upper=upper)
        ratio   = train_pol.loc[mask_g, "premium_new"] / (sub["premium"] + 1e-9)
        train_pol.loc[mask_g, "prem_base_new"]  = sub["prem_base"] * ratio
        lr_new  = sub["claim_amount"].sum() / train_pol.loc[mask_g, "prem_base_new"].sum()
        print(f"  Round {recalib_round+1} | Группа {grp}: LR {lr_g:.2%} -> {lr_new:.2%}")

    train_pol["group"] = np.where(
        train_pol["premium_new"] <= train_pol["premium"] * GROUP1_THRESHOLD, 1, 2
    )

lr_check = train_pol["claim_amount"].sum() / train_pol["prem_base_new"].sum()
if abs(lr_check - TARGET_LR) > 0.002:
    final_adj = lr_check / TARGET_LR
    for grp in [1, 2]:
        mask_g = train_pol["group"] == grp
        new_pn = (train_pol.loc[mask_g, "premium_new"] * final_adj).clip(
            upper=train_pol.loc[mask_g, "premium"] * 3
        )
        train_pol.loc[mask_g, "premium_new"] = new_pn
        ratio = new_pn / (train_pol.loc[mask_g, "premium"] + 1e-9)
        train_pol.loc[mask_g, "prem_base_new"] = train_pol.loc[mask_g, "prem_base"] * ratio
    print(f"  Финальная поправка: ×{final_adj:.4f}")


# ── Проверка ограничений кейса ─────────────────────────────────────────────
# Снижение ≤ 100%: premium_new >= 0  (lower_bound = premium * 0.0)
# Повышение ≤ 3x:  premium_new <= premium * 3  (upper_bound = premium * 3)

MIN_PREMIUM = train_pol["premium"].median() * 0.0  # = 0, соответствует "снижение ≤ 100%"
MIN_PREMIUM = max(MIN_PREMIUM, 1.0)  # абсолютный минимум 1 тг
train_pol["premium_new"] = train_pol["premium_new"].clip(lower=MIN_PREMIUM)
ratio_floor = train_pol["premium_new"] / (train_pol["premium"] + 1e-9)
train_pol["prem_base_new"] = train_pol["prem_base"] * ratio_floor

# Финальная проверка ограничений
max_increase = (train_pol["premium_new"] / train_pol["premium"]).max()
min_ratio    = (train_pol["premium_new"] / train_pol["premium"]).min()
print(f"  Проверка ограничений:")
print(f"  Макс. увеличение: {max_increase:.2f}x  {'✅' if max_increase <= 3.01 else '❌'} (лимит ×3)")
print(f"  Мин. отношение:   {min_ratio:.4f}x  {'✅' if min_ratio >= 0 else '❌'} (снижение ≤100%)")
print(f"  Минимальная премия (floor): {MIN_PREMIUM:,.0f} тг")

lr_final_total = train_pol["claim_amount"].sum() / train_pol["prem_base_new"].sum()
lr_g1 = (train_pol.loc[train_pol["group"]==1,"claim_amount"].sum() /
         train_pol.loc[train_pol["group"]==1,"prem_base_new"].sum())
lr_g2 = (train_pol.loc[train_pol["group"]==2,"claim_amount"].sum() /
         train_pol.loc[train_pol["group"]==2,"prem_base_new"].sum())
print(f"\n  После калибровки по группам:")
print(f"  Общий LR:   {lr_final_total:.2%}")
print(f"  Группа 1:   {lr_g1:.2%} | {(train_pol['group']==1).sum():,} полисов "
      f"({(train_pol['group']==1).mean():.1%})")
print(f"  Группа 2:   {lr_g2:.2%} | {(train_pol['group']==2).sum():,} полисов "
      f"({(train_pol['group']==2).mean():.1%})")

scale_g1 = (train_pol.loc[train_pol["group"]==1,"premium_new"].sum() /
            (train_pol.loc[train_pol["group"]==1,"expected_loss"].sum() / TARGET_LR + 1e-9))
scale_g2 = (train_pol.loc[train_pol["group"]==2,"premium_new"].sum() /
            (train_pol.loc[train_pol["group"]==2,"expected_loss"].sum() / TARGET_LR + 1e-9))
print(f"\n  scale_g1={scale_g1:.4f} | scale_g2={scale_g2:.4f}")

# ──────────────────────────────────────────────
# 12. МЕТРИКИ
# ──────────────────────────────────────────────
print("\n12. Метрики коэффициента выплат")
lr_after = train_pol["claim_amount"].sum() / train_pol["prem_base_new"].sum()

for grp in [1, 2]:
    sub   = train_pol[train_pol["group"] == grp]
    lr    = sub["claim_amount"].sum() / sub["prem_base_new"].sum()
    pct   = len(sub) / len(train_pol)
    avg_d = ((sub["premium_new"] / (sub["premium"] + 1e-9)) - 1).mean()
    flag  = "✅" if abs(lr - TARGET_LR) < 0.05 else "⚠️"
    arrow = "↓ снижение" if avg_d < 0 else "↑ повышение"
    print(f"  Группа {grp}: LR={lr:.2%} {flag}  |  {len(sub):,} полисов ({pct:.1%})"
          f"  |  avg Δ={avg_d:+.1%}  {arrow}")

print(f"\n  LR ДО:    {LR_BEFORE:.2%}")
print(f"  LR ПОСЛЕ: {lr_after:.2%}  (цель: {TARGET_LR:.0%})")

# ──────────────────────────────────────────────
# 13. РЕЗУЛЬТАТЫ ДЛЯ ТЕСТА
# ──────────────────────────────────────────────
print("\n13. Формирование test результатов")

test["expected_loss"] = test["prob_claim"] * test["expected_severity"]

test_pol = (
    test.groupby("contract_number", as_index=False)
        .agg(
            premium        =("premium",        "first"),
            premium_wo_term=("premium_wo_term","first"),
            prob_claim     =("prob_claim",      "first"),
            expected_loss  =("expected_loss",   "first"),
        )
)

test_pol["prem_base"] = np.where(
    test_pol["premium_wo_term"] > 0,
    test_pol["premium_wo_term"],
    test_pol["premium"]
)

raw_new = test_pol["expected_loss"] * scale / TARGET_LR
test_pol["premium_new"] = raw_new.clip(upper=test_pol["premium"] * 3)

test_pol["group"] = np.where(
    test_pol["premium_new"] <= test_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)

for grp, sc in [(1, scale_g1), (2, scale_g2)]:
    mask_g = test_pol["group"] == grp
    raw_g  = test_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
    test_pol.loc[mask_g, "premium_new"] = raw_g.clip(
        upper=test_pol.loc[mask_g, "premium"] * 3
    )

ratio_t = test_pol["premium_new"] / (test_pol["premium"] + 1e-9)
test_pol["prem_base_new"] = test_pol["prem_base"] * ratio_t
test_pol["premium_new"]   = test_pol["premium_new"].clip(lower=MIN_PREMIUM)
ratio_floor_t = test_pol["premium_new"] / (test_pol["premium"] + 1e-9)
test_pol["prem_base_new"] = test_pol["prem_base"] * ratio_floor_t

test_pol["pred_loss_ratio"] = (
    test_pol["expected_loss"] / (test_pol["prem_base"] + 1e-9)
).clip(0, 5)

result_df = test_pol[[
    "contract_number", "prob_claim", "pred_loss_ratio", "premium_new"
]].rename(columns={"prob_claim": "prob_dtp"})

# Предварительное сохранение (будет перезаписано после holdout-калибровки)
result_df.to_csv(OUTPUT_DIR / "results_test_v6_preliminary.csv", index=False)

# ──────────────────────────────────────────────
# 13b. HOLDOUT VALIDATION
# ──────────────────────────────────────────────
# Применяем обученную модель к holdout (20% train с реальным claim_amount).

print("\n13b. Holdout Validation (LR на отложенной 20% выборке)")

# Применяем ту же очистку что и к train (car_year, bonus_malus и т.д.)
for df in [train_holdout]:
    df["bonus_malus_clean"] = df["bonus_malus"].apply(clean_bonus_malus)
    df["experience_year"]   = df["experience_year"].apply(clean_experience_year)
    df["car_year"]          = df["car_year"].apply(clean_car_year)
    df["car_age"]           = df["car_age"].apply(parse_car_age)
    df["engine_volume"] = pd.to_numeric(df["engine_volume"], errors="coerce")
    df["engine_power"]  = pd.to_numeric(df["engine_power"],  errors="coerce")
    df.loc[df["engine_volume"] < 0.3,  "engine_volume"] = np.nan
    df.loc[df["engine_volume"] > 10,   "engine_volume"] = np.nan
    df.loc[df["engine_power"]  < 5,    "engine_power"]  = np.nan
    df.loc[df["engine_power"]  > 1000, "engine_power"]  = np.nan
    df["engine_volume"] = df["engine_volume"].fillna(df["engine_volume"].median())
    df["engine_power"]  = df["engine_power"].fillna(df["engine_power"].median())
    df["premium"] = pd.to_numeric(df["premium"], errors="coerce")
    df.loc[df["premium"] <= 0, "premium"] = np.nan
    df["premium"] = df["premium"].fillna(df["premium"].median())
    df["premium_wo_term"] = pd.to_numeric(df["premium_wo_term"], errors="coerce")
    df["premium_wo_term"] = df["premium_wo_term"].fillna(df["premium"] * 0.85)

train_holdout["claim_amount"] = pd.to_numeric(
    train_holdout["claim_amount"], errors="coerce"
).fillna(0)

# SCORE агрегация для holdout (те же группы что на train)
for prefix, cols in score_groups.items():
    available = [c for c in cols if c in train_holdout.columns]
    if available:
        train_holdout[f"{prefix}_avg"] = (
            train_holdout[available].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        )
    else:
        train_holdout[f"{prefix}_avg"] = 0.0
# Удаляем исходные SCORE из holdout
orig_score_cols = [c for c in train_holdout.columns
                   if c.startswith("SCORE") and not c.endswith("_avg")]
train_holdout.drop(columns=orig_score_cols, inplace=True, errors="ignore")

# Кодируем категориальные так же как train
for col in cat_cols:
    if col in train_holdout.columns:
        train_holdout[col] = le_dict[col].transform(
            train_holdout[col].astype(str)
        )

# Feature engineering для holdout
train_holdout = feature_engineering(train_holdout)
train_holdout = train_holdout.merge(policy_feat_agg, on="contract_number", how="left")
for col in POL_FEAT_NAMES:
    train_holdout[col] = train_holdout[col].fillna(policy_feat_agg[col].mean())

X_holdout = train_holdout[feature_cols].apply(pd.to_numeric, errors="coerce")
train_holdout["prob_claim"]        = calibrate(model_freq.predict(X_holdout), CLAIM_RATE)
train_holdout["pred_log_severity"] = model_sev.predict(X_holdout)
train_holdout["expected_severity"] = np.expm1(
    train_holdout["pred_log_severity"] + log_bias
).clip(lower=min_sev)
# Применяем тот же региональный коэффициент что и на train/test
train_holdout["expected_severity"] = apply_region_factor(
    train_holdout, region_factor_map
).clip(lower=min_sev)
train_holdout["expected_loss"] = (
    train_holdout["prob_claim"] * train_holdout["expected_severity"]
)
train_holdout["claim_amount"] = pd.to_numeric(
    train_holdout["claim_amount"], errors="coerce"
).fillna(0)
train_holdout["premium_wo_term"] = pd.to_numeric(
    train_holdout["premium_wo_term"], errors="coerce"
).fillna(train_holdout["premium"] * 0.85)

holdout_pol = (
    train_holdout.groupby("contract_number", as_index=False)
    .agg(
        premium        =("premium",        "first"),
        premium_wo_term=("premium_wo_term","first"),
        claim_amount   =("claim_amount",   "first"),
        expected_loss  =("expected_loss",  "first"),
    )
)
holdout_pol["prem_base"] = np.where(
    holdout_pol["premium_wo_term"] > 0,
    holdout_pol["premium_wo_term"],
    holdout_pol["premium"]
)

# Применяем те же scale_g1 / scale_g2 что на train
raw_h = holdout_pol["expected_loss"] * scale / TARGET_LR
holdout_pol["premium_new"] = raw_h.clip(upper=holdout_pol["premium"] * 3)
holdout_pol["group"] = np.where(
    holdout_pol["premium_new"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1), (2, scale_g2)]:
    mask_g = holdout_pol["group"] == grp
    raw_g  = holdout_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
    holdout_pol.loc[mask_g, "premium_new"] = raw_g.clip(
        upper=holdout_pol.loc[mask_g, "premium"] * 3
    )
holdout_pol["premium_new"] = holdout_pol["premium_new"].clip(lower=MIN_PREMIUM)
ratio_h = holdout_pol["premium_new"] / (holdout_pol["premium"] + 1e-9)
holdout_pol["prem_base_new"] = holdout_pol["prem_base"] * ratio_h

lr_holdout = holdout_pol["claim_amount"].sum() / holdout_pol["prem_base_new"].sum()

# ── Диагностика аномалий в holdout ────────────────────────────────────────
claim_holdout = holdout_pol[holdout_pol["claim_amount"] > 0]["claim_amount"]
total_claims_h = holdout_pol["claim_amount"].sum()
top5_claims    = claim_holdout.nlargest(5).sum()
top5_pct       = top5_claims / total_claims_h * 100
p95_claim      = claim_holdout.quantile(0.95)
print(f"\n  Диагностика holdout выплат:")
print(f"  Всего выплат:           {total_claims_h:,.0f} тг")
print(f"  Топ-5 выплат сумма:     {top5_claims:,.0f} тг ({top5_pct:.1f}% от всех)")
print(f"  95-й перцентиль:        {p95_claim:,.0f} тг")
print(f"  Медиана выплат:         {claim_holdout.median():,.0f} тг")
# Если топ-5 > 30% от суммы - есть аномалии
if top5_pct > 30:
    print(f"  ⚠️  Топ-5 выплат составляют {top5_pct:.1f}% - возможны аномалии")
    # LR без топ-5 выплат
    top5_idx = claim_holdout.nlargest(5).index
    claims_no_top5 = holdout_pol["claim_amount"].copy()
    claims_no_top5.loc[top5_idx] = 0
    lr_no_top5 = claims_no_top5.sum() / holdout_pol["prem_base_new"].sum()
    print(f"  LR holdout без топ-5:   {lr_no_top5:.2%}")
else:
    print(f"  ✅ Аномалий нет - LR={lr_holdout:.2%} отражает реальный риск")
lr_holdout_g1 = (
    holdout_pol.loc[holdout_pol["group"]==1, "claim_amount"].sum() /
    holdout_pol.loc[holdout_pol["group"]==1, "prem_base_new"].sum()
) if (holdout_pol["group"]==1).any() else 0
lr_holdout_g2 = (
    holdout_pol.loc[holdout_pol["group"]==2, "claim_amount"].sum() /
    holdout_pol.loc[holdout_pol["group"]==2, "prem_base_new"].sum()
) if (holdout_pol["group"]==2).any() else 0
pct_g1_h = (holdout_pol["group"]==1).mean()

delta_lr = abs(lr_holdout - lr_after)
flag_h   = "✅" if delta_lr < 0.05 else "❌ ПЕРЕОБУЧЕНИЕ"

print(f"  LR на train   (80%):          {lr_after:.2%}")
print(f"  LR на holdout (20%):          {lr_holdout:.2%}  {flag_h}")
print(f"  Δ между ними:                 {delta_lr:.2%}")
print(f"  Группа 1 holdout: LR={lr_holdout_g1:.2%} | доля={pct_g1_h:.1%}")
print(f"  Группа 2 holdout: LR={lr_holdout_g2:.2%}")

# ── ПЕРЕКАЛИБРОВКА SCALE ПО HOLDOUT ──────────────────────────────────────
# Калибруем: 1) общий scale_h, 2) групповые scale_g1h/scale_g2h по holdout.
print(f"\n  Перекалибровка scale по holdout...")

delta_holdout = abs(lr_holdout - TARGET_LR)
if delta_holdout <= 0.05:
    scale_h = scale
    lr_hc   = lr_holdout
    print(f"  ✅ Holdout LR={lr_holdout:.2%} уже близко к цели - scale не меняем")
    print(f"  scale (применён): {scale_h:.4f}")
else:
    scale_h = scale
    for i in range(30):
        raw_hc   = holdout_pol["expected_loss"] * scale_h / TARGET_LR
        pn_hc    = raw_hc.clip(upper=holdout_pol["premium"] * 3)
        ratio_hc = pn_hc / (holdout_pol["premium"] + 1e-9)
        pb_hc    = (holdout_pol["prem_base"] * ratio_hc).clip(lower=MIN_PREMIUM)
        lr_hc    = holdout_pol["claim_amount"].sum() / (pb_hc.sum() + 1e-9)
        if abs(lr_hc - TARGET_LR) < 0.001:
            break
        scale_h *= (lr_hc / TARGET_LR)
    print(f"  scale (train): {scale:.4f}  ->  scale (holdout): {scale_h:.4f}")
    print(f"  LR holdout после перекалибровки: {lr_hc:.2%}")

# ── ГРУППОВАЯ ПЕРЕКАЛИБРОВКА ПО HOLDOUT ───────────────────────────────────
# Дополнительно калибруем scale_g1/scale_g2 по holdout чтобы
# LR ~70% выполнялось в каждой группе на holdout.
print(f"\n  Групповая перекалибровка по holdout...")

# Определяем группы на holdout с новым scale_h
raw_h2 = holdout_pol["expected_loss"] * scale_h / TARGET_LR
holdout_pol["premium_new_h"] = raw_h2.clip(upper=holdout_pol["premium"] * 3)
holdout_pol["group_h"] = np.where(
    holdout_pol["premium_new_h"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)

# Калибруем scale_g1h и scale_g2h итеративно по holdout
adj = scale_h / scale
scale_g1h = scale_g1 * adj
scale_g2h = scale_g2 * adj

for recalib_round in range(3):
    for grp, sc_attr in [(1, "scale_g1h"), (2, "scale_g2h")]:
        mask_g = holdout_pol["group_h"] == grp
        if not mask_g.any():
            continue
        sc = scale_g1h if grp == 1 else scale_g2h
        raw_g = holdout_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
        pn_g  = raw_g.clip(upper=holdout_pol.loc[mask_g, "premium"] * 3)
        pn_g  = pn_g.clip(lower=MIN_PREMIUM)
        ratio_g = pn_g / (holdout_pol.loc[mask_g, "premium"] + 1e-9)
        pb_g  = holdout_pol.loc[mask_g, "prem_base"] * ratio_g
        lr_g  = holdout_pol.loc[mask_g, "claim_amount"].sum() / (pb_g.sum() + 1e-9)
        if grp == 1:
            scale_g1h *= (lr_g / TARGET_LR)
        else:
            scale_g2h *= (lr_g / TARGET_LR)

# Финальная проверка групп на holdout
for grp, sc in [(1, scale_g1h), (2, scale_g2h)]:
    mask_g = holdout_pol["group_h"] == grp
    if not mask_g.any():
        continue
    raw_g = holdout_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
    pn_g  = raw_g.clip(upper=holdout_pol.loc[mask_g, "premium"] * 3).clip(lower=MIN_PREMIUM)
    ratio_g = pn_g / (holdout_pol.loc[mask_g, "premium"] + 1e-9)
    pb_g  = holdout_pol.loc[mask_g, "prem_base"] * ratio_g
    lr_g  = holdout_pol.loc[mask_g, "claim_amount"].sum() / (pb_g.sum() + 1e-9)
    print(f"  Группа {grp} holdout: LR={lr_g:.2%} | доля={mask_g.mean():.1%}")

print(f"  scale_g1h={scale_g1h:.4f} | scale_g2h={scale_g2h:.4f}")

# Перегенерируем test с holdout-откалиброванными scale
print(f"  Применяем holdout scale к тестовой выборке...")
raw_t2 = test_pol["expected_loss"] * scale_h / TARGET_LR
test_pol["premium_new"] = raw_t2.clip(upper=test_pol["premium"] * 3)
test_pol["group"] = np.where(
    test_pol["premium_new"] <= test_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1h), (2, scale_g2h)]:
    mask_g = test_pol["group"] == grp
    raw_g2 = test_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
    test_pol.loc[mask_g, "premium_new"] = raw_g2.clip(
        upper=test_pol.loc[mask_g, "premium"] * 3
    )
test_pol["premium_new"] = test_pol["premium_new"].clip(lower=MIN_PREMIUM)
ratio_t2 = test_pol["premium_new"] / (test_pol["premium"] + 1e-9)
test_pol["prem_base_new"] = test_pol["prem_base"] * ratio_t2
test_pol["pred_loss_ratio"] = (
    test_pol["expected_loss"] / (test_pol["prem_base"] + 1e-9)
).clip(0, 5)

result_df = test_pol[[
    "contract_number", "prob_claim", "pred_loss_ratio", "premium_new"
]].rename(columns={"prob_claim": "prob_dtp"})
result_df.to_csv(OUTPUT_DIR / "results_test_v6.csv", index=False)
print(f"  ✅ results_test_v6.csv перезаписан с holdout-откалиброванным scale")
print(f"  premium_new test: median={result_df['premium_new'].median():,.0f} тг | "
      f"mean={result_df['premium_new'].mean():,.0f} тг")

# Финальные holdout метрики с holdout-откалиброванными групповыми scale
raw_hf = holdout_pol["expected_loss"] * scale_h / TARGET_LR
holdout_pol["premium_new_final"] = raw_hf.clip(upper=holdout_pol["premium"] * 3)
holdout_pol["group_final"] = np.where(
    holdout_pol["premium_new_final"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1h), (2, scale_g2h)]:
    mask_g = holdout_pol["group_final"] == grp
    raw_gf = holdout_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
    holdout_pol.loc[mask_g, "premium_new_final"] = raw_gf.clip(
        upper=holdout_pol.loc[mask_g, "premium"] * 3
    )
holdout_pol["premium_new_final"] = holdout_pol["premium_new_final"].clip(lower=MIN_PREMIUM)
ratio_hf = holdout_pol["premium_new_final"] / (holdout_pol["premium"] + 1e-9)
holdout_pol["prem_base_new_final"] = holdout_pol["prem_base"] * ratio_hf
holdout_pol["group_final"] = np.where(
    holdout_pol["premium_new_final"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
lr_holdout_final = holdout_pol["claim_amount"].sum() / holdout_pol["prem_base_new_final"].sum()
pct_g1_h = (holdout_pol["group_final"] == 1).mean()
delta_lr = abs(lr_holdout_final - TARGET_LR)
flag_h = "✅" if delta_lr < 0.05 else "❌"
print(f"\n  LR holdout (финальный):       {lr_holdout_final:.2%}  {flag_h}")
print(f"  Δ от цели 70%:                {delta_lr:.2%}")

# ──────────────────────────────────────────────
# 13c. ТЕСТ НА УСТОЙЧИВОСТЬ К ШУМУ
# ──────────────────────────────────────────────

# Проверяем: добавляем 1% шум к признакам и смотрим корреляцию предсказаний.
print("\n13c. Тест на устойчивость к шуму")

np.random.seed(42)
X_noise     = X + np.random.normal(0, 0.01 * X.std(), X.shape)
pred_orig   = model_freq.predict(X)
pred_noisy  = model_freq.predict(X_noise)
corr_noise  = np.corrcoef(pred_orig, pred_noisy)[0, 1]

# LR при зашумлённых предсказаниях
prob_noise  = calibrate(pred_noisy, CLAIM_RATE)
train["prob_noise"]    = prob_noise
train["exp_loss_noise"] = train["prob_noise"] * train["expected_severity"]
train_pol_noise = policy_agg(
    train,
    extra=dict(exp_loss_noise=("exp_loss_noise","first"))
)
train_pol_noise["prem_base"] = np.where(
    train_pol_noise["premium_wo_term"] > 0,
    train_pol_noise["premium_wo_term"],
    train_pol_noise["premium"]
)
raw_n = train_pol_noise["exp_loss_noise"] * scale / TARGET_LR
train_pol_noise["premium_new"] = raw_n.clip(upper=train_pol_noise["premium"] * 3)
train_pol_noise["premium_new"] = train_pol_noise["premium_new"].clip(lower=MIN_PREMIUM)
ratio_n = train_pol_noise["premium_new"] / (train_pol_noise["premium"] + 1e-9)
train_pol_noise["prem_base_new"] = train_pol_noise["prem_base"] * ratio_n
lr_noise = train_pol_noise["claim_amount"].sum() / train_pol_noise["prem_base_new"].sum()

flag_noise = "✅ устойчива" if corr_noise >= 0.95 else "⚠️ чувствительна к шуму"
print(f"  Корреляция предсказаний (оригинал vs шум 1%): {corr_noise:.4f}  {flag_noise}")
print(f"  LR при зашумлённых признаках:                 {lr_noise:.2%}")
print(f"  Δ LR (оригинал vs шум):                       {abs(lr_noise - lr_after):.2%}")

# Убираем временные колонки
train.drop(columns=["prob_noise","exp_loss_noise"], inplace=True, errors="ignore")

# ──────────────────────────────────────────────
# ИТОГОВЫЙ ОТЧЁТ
# ──────────────────────────────────────────────
W = 62
def row(label, val, note=""):
    line = f"  {label:<28} {val:>20}"
    if note: line += f"  {note}"
    print(line)
def divider(): print("  " + "─" * W)
def header(title):
    print("\n  ┌" + "─"*W + "┐")
    print("  │" + title.center(W) + "│")
    print("  └" + "─"*W + "┘")

mean_auc = np.mean(auc_list)
mean_atr = np.mean(auc_train_list)

header("A. ПРОВЕРКА НА ПЕРЕОБУЧЕНИЕ")
print(f"  {'Фолд':<8} {'AUC train':>12} {'AUC val':>12} {'GAP':>10} {'best_iter':>12}")
divider()
for i, (atr, avl, bi) in enumerate(zip(auc_train_list, auc_list, best_iters)):
    gap = atr - avl
    flag = "✅" if gap < 0.05 else ("⚠️" if gap < 0.10 else ("⚠️" if gap < 0.15 else "❌"))
    print(f"  {i+1:<8} {atr:>12.4f} {avl:>12.4f} {gap:>+10.4f} {bi:>12}  {flag}")
divider()
gap_verdict = ("✅ GAP < 0.05" if mean_gap < 0.05 else
               "✅ GAP 0.05–0.10" if mean_gap < 0.10 else
               "⚠️  GAP 0.10–0.15" if mean_gap < 0.15 else
               "❌ GAP > 0.15")
print(f"  {'Среднее':<8} {mean_atr:>12.4f} {mean_auc:>12.4f} {mean_gap:>+10.4f}")
print(f"\n  Вердикт: {gap_verdict}")

header("B. МЕТРИКИ МОДЕЛЕЙ")
print(f"\n  ── Frequency Model (LightGBM)")
divider()
row("CV AUC (val)",      f"{mean_auc:.4f} ± {np.std(auc_list):.4f}")
row("GINI",              f"{2*mean_auc-1:.4f}")
row("CV AUC (train)",    f"{mean_atr:.4f}")
row("Train-val GAP",     f"{mean_gap:+.4f}", gap_verdict)
row("Калибровка train",  f"{train['prob_claim'].mean():.5f}", f"цель={CLAIM_RATE:.5f}")
row("Калибровка test",   f"{test['prob_claim'].mean():.5f}",  f"цель={CLAIM_RATE:.5f}")
print(f"\n  ── Severity Model")
divider()
row("R²",                f"{r2:.4f}")
row("Log-bias",          f"{log_bias:.4f}")

header("D. ЦЕНООБРАЗОВАНИЕ - TRAIN")
divider()
row("LR ДО",             f"{LR_BEFORE:.2%}")
row("LR ПОСЛЕ",          f"{lr_after:.2%}", "<- цель 70%  " + ("✅" if abs(lr_after-TARGET_LR)<0.005 else "❌"))
row("Scale factor",      f"{scale:.4f}")

header("F. TRAIN vs TEST - СРАВНЕНИЕ")
print(f"  {'Метрика':<28} {'Train':>12} {'Test':>12} {'Δ':>10}  Статус")
divider()
checks = [
    ("prob_claim mean",         train["prob_claim"].mean(),             test["prob_claim"].mean(),             0.002),
    ("prob_claim std",          train["prob_claim"].std(),              test["prob_claim"].std(),              0.010),
    ("expected_severity mean",  train["expected_severity"].mean(),      test["expected_severity"].mean(),
     train["expected_severity"].mean()*0.10),
    ("premium_new mean",        train_pol["premium_new"].mean(),        test_pol["premium_new"].mean(),
     train_pol["premium_new"].mean()*0.25),
    ("pred_loss_ratio mean",
     (train["expected_loss"] / train_pol.set_index("contract_number")
      .reindex(train["contract_number"].values)["prem_base"].values).clip(0,5).mean(),
     test_pol["pred_loss_ratio"].mean(), 0.15),
]
all_ok = True
for label, tr_v, te_v, tol in checks:
    delta = abs(tr_v - te_v); ok = delta <= tol
    if not ok: all_ok = False
    print(f"  {label:<28} {tr_v:>12.4f} {te_v:>12.4f} {delta:>10.4f}  {'✅' if ok else '⚠️'}")
divider()
print(f"  Итог: {'✅ Распределения схожи' if all_ok else '⚠️  Есть расхождения'}")

header("G. ПРОВЕРКА ОГРАНИЧЕНИЙ")
divider()
max_ok    = (test_pol["premium_new"] <= test_pol["premium"] * 3 + 0.01).all()
lr_ok     = abs(lr_after - TARGET_LR) < 0.01
cal_train = abs(train["prob_claim"].mean() - CLAIM_RATE) < 0.0005
cal_test  = abs(test["prob_claim"].mean()  - CLAIM_RATE) < 0.0005
print(f"  {'Ограничение сверху ×3:':<35} {'✅' if max_ok    else '❌'}")
print(f"  {'LR цель 70%:':<35} {'✅' if lr_ok     else '❌'}  (факт: {lr_after:.2%})")
print(f"  {'Калибровка train:':<35} {'✅' if cal_train else '❌'}  ({train['prob_claim'].mean():.5f})")
print(f"  {'Калибровка test:':<35} {'✅' if cal_test  else '❌'}  ({test['prob_claim'].mean():.5f})")

header("H. HOLDOUT VALIDATION (ключевой тест)")
divider()
flag_h2 = "✅" if delta_lr < 0.05 else "❌"
row("LR на train (80%)",           f"{lr_after:.2%}")
row("LR на holdout до калибровки", f"{lr_holdout:.2%}", "❌" if abs(lr_holdout-TARGET_LR)>0.05 else "✅")
row("LR на holdout после калибр.", f"{lr_holdout_final:.2%}", flag_h2)
row("Δ от цели 70%",               f"{delta_lr:.2%}", "цель < 5%")
row("scale train",                 f"{scale:.4f}")
row("scale holdout (применён)",    f"{scale_h:.4f}")
row("scale_g1h",                   f"{scale_g1h:.4f}")
row("scale_g2h",                   f"{scale_g2h:.4f}")
row("Группа 1 holdout доля",       f"{pct_g1_h:.1%}")
print(f"\n  Вердикт: {'✅ Scale откалиброван по holdout - результат у судей ожидаемо ~70%' if delta_lr < 0.05 else '❌ Требуется дополнительная работа'}")

header("I. ТЕСТ НА ШУМОУСТОЙЧИВОСТЬ")
divider()
row("Корреляция (orig vs +1% шум)", f"{corr_noise:.4f}", "✅ устойчива" if corr_noise >= 0.95 else "⚠️ чувствительна")
row("LR при зашумлённых признаках", f"{lr_noise:.2%}")
row("Δ LR (orig vs шум)",           f"{abs(lr_noise-lr_after):.2%}", "цель < 3%")

# ──────────────────────────────────────────────
# 14. СОХРАНЕНИЕ
# ──────────────────────────────────────────────
model_freq.save_model(str(OUTPUT_DIR / "model_frequency_v6.txt"))
model_sev.save_model(str(OUTPUT_DIR / "model_severity_v6.txt"))
joblib.dump(le_dict,      OUTPUT_DIR / "label_encoders_v6.pkl")
joblib.dump(feature_cols, OUTPUT_DIR / "feature_cols_v6.pkl")
joblib.dump({"scale": scale, "log_bias": log_bias, "min_sev": min_sev},
            OUTPUT_DIR / "pricing_params_v6.pkl")
print("\n14. Модели сохранены в output/")

# ──────────────────────────────────────────────
# 15. FEATURE IMPORTANCE
# ──────────────────────────────────────────────
fi = pd.DataFrame({
    "feature":    model_freq.feature_name(),
    "importance": model_freq.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(25)
print("\n15. Feature Importance (Top 25):")
print(fi.to_string(index=False))
fi.to_csv(OUTPUT_DIR / "feature_importance_v6.csv", index=False)

# ──────────────────────────────────────────────
# 16. ВИЗУАЛИЗАЦИИ
# ──────────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc as sk_auc

    fig_dir = OUTPUT_DIR / "figures"
    fig_dir.mkdir(exist_ok=True)

    fpr, tpr, _ = roc_curve(y_freq, oof_prob)
    roc_auc_val = sk_auc(fpr, tpr)
    plt.figure(figsize=(7,6))
    plt.plot(fpr, tpr, color="darkorange", lw=2,
             label=f"ROC (AUC={roc_auc_val:.4f}, GINI={2*roc_auc_val-1:.4f})")
    plt.plot([0,1],[0,1], color="navy", lw=1, linestyle="--", label="Random")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - LightGBM v6 (OOF)")
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig(fig_dir / "roc_curve.png", dpi=150); plt.close()

    fig, axes = plt.subplots(1,2, figsize=(14,5))
    axes[0].hist(train["prob_claim"][y_freq==0], bins=60, alpha=0.5,
                 label="No claim", density=True, color="steelblue")
    axes[0].hist(train["prob_claim"][y_freq==1], bins=60, alpha=0.5,
                 label="Claim", density=True, color="orange")
    axes[0].set_title("Распределение вероятностей ДТП"); axes[0].legend()
    axes[1].hist(train["prob_claim"], bins=80, edgecolor="white", color="steelblue")
    axes[1].set_yscale("log"); axes[1].set_title("Общее распределение (log scale)")
    plt.tight_layout()
    plt.savefig(fig_dir / "prob_distribution.png", dpi=150); plt.close()

    fi_plot = fi.head(20)
    plt.figure(figsize=(9,6))
    colors_fi = ["orange" if "SCORE" in f else "steelblue" for f in fi_plot["feature"]]
    plt.barh(fi_plot["feature"], fi_plot["importance"],
             color=colors_fi[::-1], edgecolor="white")
    plt.xlabel("Feature importance (gain)")
    plt.title("Top 20 features - LightGBM v6")
    plt.gca().invert_yaxis(); plt.tight_layout()
    plt.savefig(fig_dir / "feature_importance.png", dpi=150); plt.close()

    fig, axes = plt.subplots(1,2, figsize=(14,6))
    groups  = ["До", "После", "Группа 1", "Группа 2"]
    lr_vals = [LR_BEFORE*100, lr_after*100, lr_g1*100, lr_g2*100]
    colors_lr = ["red", "green", "green", "steelblue"]
    bars = axes[0].bar(groups, lr_vals, color=colors_lr, edgecolor="white")
    axes[0].axhline(y=70, color="black", linestyle="--", label="Цель 70%")
    for bar, v in zip(bars, lr_vals):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                     f"{v:.1f}%", ha="center")
    axes[0].set_ylabel("Loss Ratio (%)"); axes[0].set_title("LR до и после"); axes[0].legend()
    grp_counts = train_pol["group"].value_counts().sort_index()
    g1 = grp_counts.get(1,0); g2 = grp_counts.get(2,0)
    axes[1].pie([g1,g2],
                labels=[f"Группа 1\n{g1:,}", f"Группа 2\n{g2:,}"],
                colors=["green","tomato"], autopct="%1.1f%%", startangle=90,
                wedgeprops={"edgecolor":"white"})
    axes[1].set_title("Распределение полисов")
    plt.tight_layout()
    plt.savefig(fig_dir / "loss_ratio.png", dpi=150); plt.close()

    sample = train_pol.sample(min(5000, len(train_pol)), random_state=42)
    plt.figure(figsize=(8,8))
    sc_colors = ["green" if g==1 else "tomato" for g in sample["group"]]
    plt.scatter(sample["premium"], sample["premium_new"], c=sc_colors, alpha=0.3, s=10)
    mx = max(sample["premium"].max(), sample["premium_new"].max())
    plt.plot([0,mx],[0,mx],   color="black", linestyle="--", label="Без изменений")
    plt.plot([0,mx],[0,mx*3], color="gray",  linestyle=":", label="×3 потолок")
    plt.xlabel("Текущая премия"); plt.ylabel("Новая премия")
    plt.title("Текущая vs Новая премия"); plt.legend(); plt.tight_layout()
    plt.savefig(fig_dir / "premium_scatter.png", dpi=150); plt.close()

    print("\n16. Визуализации сохранены в output/figures/")

except Exception as e:
    print(f"\n16. Визуализации пропущены: {e}")

# ──────────────────────────────────────────────
# 16b. CALIBRATION PLOT
# ──────────────────────────────────────────────
print("\n16b. Calibration Analysis (OOF predictions)")
try:
    from sklearn.calibration import calibration_curve
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig_dir = OUTPUT_DIR / "figures"
    fig_dir.mkdir(exist_ok=True)

    # Calibration curve - 10 бинов по квантилям вероятности
    fraction_of_positives, mean_predicted = calibration_curve(
        y_freq, train["prob_claim"], n_bins=10, strategy="quantile"
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Левый: calibration curve
    axes[0].plot(mean_predicted, fraction_of_positives,
                 "o-", color="darkorange", label="Модель")
    axes[0].plot([0, 1], [0, 1], "k--", label="Идеальная")
    axes[0].set_xlabel("Предсказанная вероятность")
    axes[0].set_ylabel("Реальная частота")
    axes[0].set_title("Calibration Plot (OOF)")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Правый: распределение вероятностей по децилям
    train["prob_decile"] = pd.qcut(train["prob_claim"], q=10, labels=False)
    decile_stats = train.groupby("prob_decile").agg(
        pred_rate=("prob_claim", "mean"),
        actual_rate=("is_claim", "mean"),
        count=("is_claim", "count")
    ).reset_index()
    x = np.arange(len(decile_stats))
    axes[1].bar(x - 0.2, decile_stats["pred_rate"] * 100, 0.4,
                label="Предсказано", color="steelblue", alpha=0.8)
    axes[1].bar(x + 0.2, decile_stats["actual_rate"] * 100, 0.4,
                label="Реально", color="orange", alpha=0.8)
    axes[1].set_xlabel("Дециль риска")
    axes[1].set_ylabel("Частота ДТП (%)")
    axes[1].set_title("Предсказанная vs Реальная частота по децилям")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(fig_dir / "calibration_plot.png", dpi=150)
    plt.close()

    # Выводим в лог
    print(f"  {'Дециль':<8} {'Предсказано':>14} {'Реально':>14} {'Отношение':>12}")
    print("  " + "─" * 52)
    for _, row_d in decile_stats.iterrows():
        ratio = row_d["actual_rate"] / (row_d["pred_rate"] + 1e-9)
        flag  = "✅" if 0.8 <= ratio <= 1.2 else "⚠️"
        print(f"  {int(row_d['prob_decile'])+1:<8} "
              f"{row_d['pred_rate']*100:>13.3f}% "
              f"{row_d['actual_rate']*100:>13.3f}% "
              f"{ratio:>11.2f}x  {flag}")

    train.drop(columns=["prob_decile"], inplace=True, errors="ignore")
    print("  Calibration plot сохранён: output/figures/calibration_plot.png")

except Exception as e:
    print(f"  Calibration plot пропущен: {e}")

# ──────────────────────────────────────────────
# 16c. АНАЛИЗ ОШИБОК
# ──────────────────────────────────────────────
print("\n16c. Анализ ошибок (где модель сильно ошибается)")
try:
    # Полисы с реальными выплатами - где expected_loss сильно занижен
    claims_df = train_pol[train_pol["claim_amount"] > 0].copy()
    claims_df["error_ratio"] = claims_df["claim_amount"] / (claims_df["expected_loss"] + 1e-9)

    # Топ-10% самых недооценённых полисов
    threshold = claims_df["error_ratio"].quantile(0.90)
    underest  = claims_df[claims_df["error_ratio"] > threshold]

    print(f"  Всего убыточных полисов: {len(claims_df):,}")
    print(f"  Порог 90-го перцентиля error_ratio: {threshold:.1f}x")
    print(f"  Сильно недооценённых (error_ratio > {threshold:.1f}x): {len(underest):,}")

    # Медиана error_ratio по регионам
    region_err = (
        claims_df.merge(
            train[["contract_number","region_id"]].drop_duplicates(),
            on="contract_number", how="left"
        )
        .groupby("region_id")["error_ratio"]
        .agg(["median","count"])
        .rename(columns={"median":"median_error","count":"n_claims"})
        .query("n_claims >= 10")
        .sort_values("median_error", ascending=False)
        .head(10)
    )
    if len(region_err) > 0:
        print(f"\n  Топ-10 регионов с наибольшей недооценкой риска:")
        print(f"  {'region_id':<12} {'median_error':>14} {'n_claims':>10}")
        print("  " + "─" * 40)
        for rid, row_r in region_err.iterrows():
            flag = "⚠️" if row_r["median_error"] > 3 else ""
            print(f"  {rid:<12} {row_r['median_error']:>13.1f}x "
                  f"{int(row_r['n_claims']):>10}  {flag}")

    print(f"\n  Вывод: если конкретные регионы систематически недооцениваются,")
    print(f"  можно добавить region-specific поправочный коэффициент.")

except Exception as e:
    print(f"  Анализ ошибок пропущен: {e}")

print("\n" + "=" * 60)
print("ГОТОВО. Все файлы в:", OUTPUT_DIR)
print("=" * 60)

1. Загрузка данных
Train: (569508, 159) | Test: (244073, 156)
  Train fit: 455,606 | Holdout (20%): 113,902

2. Дубликатов: 17 -> убраны
  Определён год датасета: 2022
  SCORE-колонок удалено (>95% нулей): 0
  SCORE-колонок осталось: 128
  SCORE сжаты: 128 -> 12 групповых средних

3. Частота выплат: 1.944%

4a. Удалено колонок >80% пропусков: 0  []
4. Очистка признаков - готово
   Пропусков после чистки: 1,156,719

5. LR ДО: 121.58% | Полисов: 177,407
6. Feature Engineering - готово
6c. Агрегаты по contract_number
  pol_driver_cnt, pol_avg_bm, pol_max_bm_risk, pol_min_exp, pol_any_new

7. IV:
              feature     IV
          pol_avg_bm 0.1506
     pol_max_bm_risk 0.1391
           region_id 0.1027
             bm_risk 0.0731
   bonus_malus_clean 0.0707
          risk_score 0.0677
         pol_min_exp 0.0594
         region_x_bm 0.0431
     vehicle_x_power 0.0379
     experience_year 0.0319
           exp_group 0.0262
         power_x_new 0.0234
        engine_power 0.0222
       